# Evologics USBL — Processed

Visualises processed Evologics USBL data from the unified output schema:
ship track and target positions on an interactive map, and time-series
of target XYZ in sensor frame vs vessel frame side-by-side.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
DEPLOYMENT: str = "qdc5ghs3_20210315_230947"

# NOTE: Deployments with Evologics USBL
# qd61g27j_20170523_040815     → 201705_silver_streak
# qdc5ghs3_20210315_230947     → 202102_poppag
# qdch0ftq_20170526_025746     → 201705_silver_streak
# qdch0ftq_20210315_034028     → 202102_poppag
# qdchdmy1_20170525_234624     → 201705_silver_streak
# qdchdmy1_20210315_081519     → 202102_poppag
# qtqxshxs_20150327_015552     → 201502_tasmania_bluefin
# qtqxshxs_20150328_000850     → 201502_tasmania_bluefin
# qtqxshxs_20150328_042551     → 201502_tasmania_bluefin

PROCESSED_DIR: Path = Path("/data/exos_01/acfr_messages_v4_telemetry_processed")
USBL_FILE: Path = PROCESSED_DIR / f"{DEPLOYMENT}_usbl_evologics.csv"

## Load data

In [ ]:
REQUIRED_COLS: list[str] = [
    "timestamp",
    "ship_latitude",
    "ship_longitude",
    "target_latitude",
    "target_longitude",
    "target_depth",
    "target_x_sensor",
    "target_y_sensor",
    "target_z_sensor",
    "target_x_vessel",
    "target_y_vessel",
    "target_z_vessel",
    "target_horizontal_range",
    "horizontal_position_std",
    "depth_position_std",
    "usbl_extrinsics_locx",
    "usbl_extrinsics_locy",
    "usbl_extrinsics_locz",
    "usbl_extrinsics_rotx",
    "usbl_extrinsics_roty",
    "usbl_extrinsics_rotz",
]

usbl: pd.DataFrame = pd.read_csv(
    USBL_FILE, parse_dates=["timestamp"], date_format="ISO8601"
)
missing: list[str] = [col for col in REQUIRED_COLS if col not in usbl.columns]
if missing:
    raise ValueError(f"CSV is missing required columns: {missing}")

extrinsics_label: str = (
    f"loc=({usbl['usbl_extrinsics_locx'].iloc[0]:.3f}, "
    f"{usbl['usbl_extrinsics_locy'].iloc[0]:.3f}, "
    f"{usbl['usbl_extrinsics_locz'].iloc[0]:.3f}) m  "
    f"rot=({np.degrees(usbl['usbl_extrinsics_rotx'].iloc[0]):.2f}, "
    f"{np.degrees(usbl['usbl_extrinsics_roty'].iloc[0]):.2f}, "
    f"{np.degrees(usbl['usbl_extrinsics_rotz'].iloc[0]):.2f}) deg"
)

print(f"Deployment  : {DEPLOYMENT}")
print(f"Rows        : {len(usbl)}")
print(f"Extrinsics  : {extrinsics_label}")
usbl.head(3)

## Overview map: ship track and target positions

In [ ]:
t_s: np.ndarray = (usbl["timestamp"].astype(np.int64) / 1e9).to_numpy()
elapsed: np.ndarray = (t_s - t_s.min()) / 60.0

center_lat: float = float(usbl["target_latitude"].mean())
center_lon: float = float(usbl["target_longitude"].mean())

colorscale: str = "Viridis"

fig_map = go.Figure()

fig_map.add_trace(
    go.Scattermapbox(
        lat=usbl["ship_latitude"],
        lon=usbl["ship_longitude"],
        mode="lines+markers",
        marker=dict(
            size=4,
            color=elapsed,
            colorscale=colorscale,
            cmin=0,
            cmax=float(elapsed.max()),
            showscale=True,
            colorbar=dict(title="Elapsed (min)", x=0.92),
        ),
        line=dict(width=1, color="royalblue"),
        name="Ship track",
        hovertemplate=(
            "<b>Ship</b><br>"
            "Lat: %{lat:.6f}<br>"
            "Lon: %{lon:.6f}<br>"
            "<extra></extra>"
        ),
    )
)

fig_map.add_trace(
    go.Scattermapbox(
        lat=usbl["target_latitude"],
        lon=usbl["target_longitude"],
        mode="markers",
        marker=dict(
            size=6,
            color=elapsed,
            colorscale=colorscale,
            cmin=0,
            cmax=float(elapsed.max()),
            showscale=False,
        ),
        name="USBL target",
        customdata=np.stack(
            [
                usbl["target_depth"],
                usbl["target_horizontal_range"],
                usbl["horizontal_position_std"],
            ],
            axis=1,
        ),
        hovertemplate=(
            "<b>USBL target</b><br>"
            "Lat: %{lat:.6f}<br>"
            "Lon: %{lon:.6f}<br>"
            "Depth: %{customdata[0]:.1f} m<br>"
            "Horizontal range: %{customdata[1]:.1f} m<br>"
            "Horizontal σ: %{customdata[2]:.2f} m<br>"
            "<extra></extra>"
        ),
    )
)

fig_map.update_layout(
    title=f"{DEPLOYMENT} (Evologics)<br><sup>{extrinsics_label}</sup>",
    mapbox=dict(
        style="open-street-map",
        center=dict(lat=center_lat, lon=center_lon),
        zoom=14,
    ),
    legend=dict(x=0.01, y=0.99),
    height=700,
    margin=dict(l=0, r=0, t=60, b=0),
)

fig_map.show()

## Time series: target XYZ — sensor frame vs vessel frame

## Time series: ship roll, pitch, and heading

In [ ]:
fig_attitude = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    row_titles=("Roll (deg)", "Pitch (deg)", "Heading (deg)"),
    vertical_spacing=0.06,
)

attitude_axes: list[tuple[str, str, int]] = [
    ("ship_roll", "steelblue", 1),
    ("ship_pitch", "darkorange", 2),
    ("ship_heading", "mediumpurple", 3),
]

for col, color, row in attitude_axes:
    fig_attitude.add_trace(
        go.Scatter(
            x=usbl["timestamp"],
            y=usbl[col],
            mode="lines+markers",
            marker=dict(size=3),
            line=dict(color=color),
            showlegend=False,
        ),
        row=row,
        col=1,
    )

fig_attitude.update_layout(
    title=f"Ship attitude — {DEPLOYMENT} (Evologics)",
    height=600,
)

fig_attitude.show()

In [ ]:
fig_xyz = make_subplots(
    rows=3,
    cols=2,
    shared_xaxes=True,
    column_titles=("Sensor frame (USBL-Frame)", "Vessel frame"),
    row_titles=("X (m)", "Y (m)", "Z (m)"),
    vertical_spacing=0.06,
    horizontal_spacing=0.08,
)

axes: list[tuple[str, str, str, int]] = [
    ("target_x_sensor", "target_x_vessel", "steelblue", 1),
    ("target_y_sensor", "target_y_vessel", "darkorange", 2),
    ("target_z_sensor", "target_z_vessel", "seagreen", 3),
]

for sensor_col, vessel_col, color, row in axes:
    for col, col_idx in [(sensor_col, 1), (vessel_col, 2)]:
        fig_xyz.add_trace(
            go.Scatter(
                x=usbl["timestamp"],
                y=usbl[col],
                mode="lines+markers",
                marker=dict(size=3),
                line=dict(color=color),
                showlegend=False,
            ),
            row=row,
            col=col_idx,
        )

fig_xyz.update_layout(
    title=f"Target XYZ — sensor frame vs vessel frame ({DEPLOYMENT})<br><sup>{extrinsics_label}</sup>",
    height=700,
)

fig_xyz.show()